## **Initial setup**

Install Bambu and required packages:

In [ ]:
!echo "deb http://ppa.launchpad.net/git-core/ppa/ubuntu $(cat /etc/os-release | grep UBUNTU_CODENAME | sed 's/.*=//g') main" >> /etc/apt/sources.list.d/git-core.list
!apt-key adv --keyserver keyserver.ubuntu.com --recv-keys A1715D88E1DF1F24
!apt-get update
!apt-get install -y --no-install-recommends build-essential ca-certificates gcc-multilib g++-multilib git libtinfo5 verilator wget
!wget https://release.bambuhls.eu/appimage/bambu-latest.AppImage
!chmod +x bambu-*.AppImage
!ln -sf $PWD/bambu-*.AppImage /bin/bambu
!ln -sf $PWD/bambu-*.AppImage /bin/panda_shell
!ln -sf $PWD/bambu-*.AppImage /bin/spider
!git clone --depth 1 --filter=blob:none --branch dev/panda --sparse https://github.com/ferrandi/PandA-bambu.git
%cd PandA-bambu
!git sparse-checkout set documentation/bambu101
%cd ..
!mv PandA-bambu/documentation/bambu101/* .

# Accelerator Interface - AXI4 Master
In this section, we explore how to guide **Bambu HLS** to synthesize **AXI4 memory-mapped interfaces** using `#pragma` directives. Each exercise demonstrates a different scenario where AXI4 ports are defined through code annotations.

AXI4 (Advanced eXtensible Interface 4) is a widely-used on-chip communication protocol developed by ARM. It supports **high-throughput, memory-mapped data transfers** and is commonly used to connect accelerators to external memory in FPGA and SoC systems.

The goal of these exercises is to understand how different pragmas affect the generation of AXI4 ports and the structure of the synthesized accelerator.

## Bambu Commands Overview

These are the command-line options we will use in this section to run [Bambu](https://github.com/ferrandi/PandA-bambu) for High-Level Synthesis and simulation of C programs.

Below is a description of each option:

<br>

<table style="font-size: 20px;">
  <thead>
    <tr>
      <th style="text-align:left; padding: 8px;">🔧 Option</th>
      <th style="text-align:left; padding: 8px;">📝 Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 8px;"><code>--generate-interface=INFER</code></td>
      <td style="padding: 8px;">Enables parsing of <code>#pragma</code> directives to define memory interfaces for the accelerator.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>--generate-tb=testbench.c</code></td>
      <td style="padding: 8px;">Specifies the C testbench file used during simulation to provide inputs and validate outputs.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>source_file.c</code></td>
      <td style="padding: 8px;">The source file containing the function to synthesize (e.g., <code>sum.c</code>).</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>--top-fname=FUNCTION</code></td>
      <td style="padding: 8px;">Defines the top-level function name to be synthetized.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>--simulate</code></td>
      <td style="padding: 8px;">Runs simulation to verify correctness.</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>-v4</code></td>
      <td style="padding: 8px;">Sets verbosity level to 4 (detailed logs of all stages).</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>--mem-delay-read=VALUE</code></td>
      <td style="padding: 8px;">Sets latency (in cycles) for external memory read accesses (e.g., <code>20</code>).</td>
    </tr>
    <tr>
      <td style="padding: 8px;"><code>--mem-delay-write=VALUE</code></td>
      <td style="padding: 8px;">Sets latency (in cycles) for external memory write accesses (e.g., <code>20</code>).</td>
    </tr>
  </tbody>
</table>


## Exercise 1

In this exercise, we will learn how to add an **AXI4** port to a function using the `#pragma HLS interface` directive. We will analyze the different clauses of the pragma used to control the interface generation, focusing on how to specify the target parameter, the type of interface, and the behavior of the memory access offset.


In [ ]:
%cd /content/Axi4/Exercise1
print("=== Source file: read.c ===\n")
!cat read.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

The following pragma is used in this exercise to specify the AXI4 interface:

```c
#pragma HLS interface port = data mode = m_axi offset = direct


| Clause                  | Meaning                                                                                                                                                            |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `#pragma HLS interface` | This is a **directive for Bambu**, used to specify how function arguments should be mapped to hardware interfaces.                    |
| `port = data`           | Specifies that the pragma applies to the function parameter named `data`.                                                                                          |
| `mode = m_axi`          | Requests synthesis of an **AXI4 memory-mapped interface** for this port. This allows external memory accesses via the AXI4 protocol.                     |
| `offset = direct`       | Indicates that the base address of the AXI4 interface is **directly derived from the input pointer**. Subsequent accesses are linear and relative to this address. |


Now, we run Bambu to simulate the accelerator behavior using the source code and testbench introduced earlier. This step will generate the AXI4 interface as specified by the pragma and verify the functionality through simulation.


In [ ]:
%cd /content/Axi4/Exercise1
!bambu --generate-interface=INFER --generate-tb=testbench.c read.c --top-fname=read --simulate -v4

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.


In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/Axi4/Exercise1/read.v')

## Exercise 2

In this exercise, we will see how using **multiple `#pragma HLS interface` directives** allows us to generate **distinct AXI4 ports**, one for each function parameter. Each pragma is applied to a different pointer, and Bambu will generate a dedicated AXI4 memory interface for each, named after the corresponding parameter. This is useful when your accelerator needs to access multiple independent memory regions in parallel.


In [ ]:
%cd /content/Axi4/Exercise2
print("=== Source file: sum.c ===\n")
!cat sum.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

The following pragmas are used in this exercise to specify the AXI4 interfaces:

```c
#pragma HLS interface port = v mode = m_axi offset = direct
#pragma HLS interface port = n mode = m_axi offset = direct


Now, we run Bambu to simulate the accelerator behavior using the updated source code and testbench. In this exercise, two AXI4 interfaces will be generated — one for each function parameter — according to the two `#pragma` directives. This setup allows Bambu to simulate independent memory accesses for each port.


In [ ]:
%cd /content/Axi4/Exercise2
!bambu --generate-interface=INFER --generate-tb=testbench.c sum.c --top-fname=sum --simulate -v4

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.


In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/Axi4/Exercise2/sum.v')

## Exercise 3

In this exercise, we extend the previous example by using the `bundle` clause in the `#pragma HLS interface` directives. Both function parameters (`v` and `n`) are assigned to the same AXI4 interface bundle named `test`. This causes Bambu to generate a **single AXI4 port** that handles both parameters, instead of creating separate ports for each. This technique is useful for grouping related memory accesses under one interface.


In [ ]:
%cd /content/Axi4/Exercise3
print("=== Source file: sum.c ===\n")
!cat sum.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

The following pragmas are used in this exercise to specify a bundled AXI4 interface:

```c
#pragma HLS interface port = v mode = m_axi offset = direct bundle = test
#pragma HLS interface port = n mode = m_axi offset = direct bundle = test


Now, we run Bambu to simulate the accelerator behavior using the source code and testbench for this exercise. Thanks to the `bundle` clause in the pragmas, Bambu will generate a **single AXI4 interface** shared by both parameters `v` and `n`.

In [ ]:
%cd /content/Axi4/Exercise3
!bambu --generate-interface=INFER --generate-tb=testbench.c sum.c --top-fname=sum --simulate -v4

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.

In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/Axi4/Exercise3/sum.v')

## Exercise 4

This exercise demonstrates a matrix multiplication using multiple AXI4 interfaces for the input and output matrices. Each pointer parameter (`a`, `b`, and `output`) is assigned to a distinct AXI4 port bundle to allow parallel memory accesses.

Additionally, we simulate realistic memory behavior by introducing read and write delays using the `--mem-delay-read` and `--mem-delay-write` options in Bambu. These delays model latency in external memory accesses, helping to evaluate performance under more practical conditions.


In [ ]:
%cd /content/Axi4/Exercise4
print("=== Source file: mmult.c ===\n")
!cat mmult.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

The following pragmas specify separate AXI4 interfaces for each matrix:

```c
#pragma HLS interface port = a mode = m_axi offset = direct bundle = gmem0
#pragma HLS interface port = b mode = m_axi offset = direct bundle = gmem1
#pragma HLS interface port = output mode = m_axi offset = direct bundle = gmem2


Each bundle name creates a distinct AXI4 port for the corresponding parameter, enabling concurrent access to the parameters.


Now, we run Bambu to simulate the matrix multiplication accelerator using the specified source code and testbench. The command includes the `--mem-delay-read=20` and `--mem-delay-write=20` options to introduce memory access latencies.

In [ ]:
%cd /content/Axi4/Exercise4
!bambu --generate-interface=INFER --generate-tb=testbench.c mmult.c --top-fname=mmult --simulate -v4 --mem-delay-read=20 --mem-delay-write=20

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.

In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/Axi4/Exercise4/mmult.v')

## Exercise 5

This exercise extends the previous matrix multiplication example by adding cache pragmas to the AXI4 interface bundles. The goal is to demonstrate how using on-chip caches can accelerate memory accesses and improve overall performance of the accelerator.

By defining cache parameters, we can control the cache size, replacement policy, write policy, and other aspects, enabling fine-tuned memory optimizations.


In [ ]:
%cd /content/Axi4/Exercise5
print("=== Source file: mmult.c ===\n")
!cat mmult.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

The following cache pragmas are applied to the AXI4 bundles:

```c
#pragma HLS cache bundle = gmem0 line_count = 16 line_size = 16 bus_size = 32 ways = 1 num_write_outstanding = 2 rep_policy = lru write_policy = wt
#pragma HLS cache bundle = gmem1 line_count = 32 line_size = 16 bus_size = 32 ways = 1 num_write_outstanding = 2 rep_policy = tree write_policy = wt
#pragma HLS cache bundle = gmem2 line_count = 16 line_size = 16 bus_size = 32 ways = 1 num_write_outstanding = 4 rep_policy = tree write_policy = wb


<table style="font-size: 20px; border-collapse: collapse;" border="1" cellpadding="8">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="text-align:left;"> Clause</th>
      <th style="text-align:left;"> Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>bundle</code></td>
      <td>Specifies the interface bundle to which the cache applies.</td>
    </tr>
    <tr>
      <td><code>line_count</code></td>
      <td>Number of cache lines in the cache.</td>
    </tr>
    <tr>
      <td><code>line_size</code></td>
      <td>Size in bytes of each cache line.</td>
    </tr>
    <tr>
      <td><code>bus_size</code></td>
      <td>Width of the data bus in bits.</td>
    </tr>
    <tr>
      <td><code>ways</code></td>
      <td>Number of ways: a value of 1 means direct-mapped cache.</td>
    </tr>
    <tr>
      <td><code>num_write_outstanding</code></td>
      <td>Number of outstanding write transactions allowed simultaneously.</td>
    </tr>
    <tr>
      <td><code>rep_policy</code></td>
      <td>Cache line replacement policy, e.g., <code>lru</code> (Least Recently Used) or <code>tree</code>.</td>
    </tr>
    <tr>
      <td><code>write_policy</code></td>
      <td>Write strategy: <code>wt</code> (write-through) or <code>wb</code> (write-back).</td>
    </tr>
  </tbody>
</table>


Now, we run Bambu to simulate the matrix multiplication accelerator with cache optimizations enabled. The cache pragmas configure on-chip caches for each AXI4 interface, potentially accelerating memory accesses. As in the previous exercise, memory read and write delays are simulated with `--mem-delay-read=20` and `--mem-delay-write=20` to model realistic latency.

In [ ]:
%cd /content/Axi4/Exercise5
!bambu --generate-interface=INFER --generate-tb=testbench.c mmult.c --top-fname=mmult --simulate -v4 --mem-delay-read=20 --mem-delay-write=20

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.

In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/Axi4/Exercise5/mmult.v')

Now check if the caches are present

In [ ]:
import re

def print_verilog_module_interface(filename, module_name="IOB_cache_axi"):
    with open(filename, 'r') as f:
        content = f.read()

    pattern = rf'module\s+{module_name}\s*\(([^;]+);'
    match = re.search(pattern, content, re.DOTALL)

    if not match:
        print(f"Module '{module_name}' not found in {filename}")
        return

    interface = match.group(0)
    print(f"Interface of the `{module_name}` module in {filename}:\n")
    print(interface)

print_verilog_module_interface('/content/Axi4/Exercise5/mmult.v')

# TrueFloat Custom Floating-point arithmetic cores
The TrueFloat library is a C-based custom floating-point arithmetic library embedded in the Bambu HLS tool.
The library offers the possibility to generate floating-point cores for aritrarily defined floating-point representations where the user may select the exponent and mantissa bitwidth, and the exponent bias. Additionally, the rounding mode can be selected between nearest even and truncate, the representation may or may not include exceptional values (i.e. Infinity, NaN), and it may support or not subnormal numbers.

Synthesized C/C++ description is automatically transformed by the HLS compiler starting from a standard float/double data type implementation through ad-hoc command line options that let the user specify the custom floating-point representation with function scope granularity.

The complete information on the available API for custom floating-point designs generation is available below:
```
--fp-exception-mode=<ieee|saturation|overflow>
    Set the soft-based exception handling mode:
            ieee    - IEEE754 standard exceptions (default)
        saturation - Inf is replaced with max value, Nan becomes undefined behaviour
        overflow  - Inf and Nan results in undefined behaviour

--fp-rounding-mode=<nearest_even|truncate>
    Set the soft-based rounding handling mode:
        nearest_even - IEEE754 standard rounding mode (default)
            truncate  - No rounding is applied

--fp-format=<func_name>*e<exp_bits>m<frac_bits>b<exp_bias><rnd_mode><exc_mode><?spec><?sign>
    Define arbitrary precision floating-point format by function (use comma separated
    list for multiple definitions). (i.e.: e8m27b-127nihs represent IEEE754 single precision FP)
        func_name - Set arbitrary floating-point format for a specific function (using
                    @ symbol here will resolve to the top function)
                    (Arbitrary floating-point format will apply to specified function
                    only, use --propagate-fp-format to extend it to called functions)
        exp_bits - Number of bits used by the exponent
        frac_bits - Number of bits used by the fractional value
        exp_bias - Bias applied to the unsigned value represented by the exponent bits
        rnd_mode - Rounding mode (exclusive option):
                        n - nearest_even: IEEE754 standard rounding mode
                        t - truncate    : no rounding is applied
        exc_mode - Exception mode (exclusive option):
                        i - ieee      : IEEE754 standard exceptions
                        a - saturation: Inf is replaced with max value, Nan becomes undefined behaviour
                        o - overflow  : Inf and Nan results in undefined behaviour
            spec   - Floating-point specialization string (multiple choice):
                        h - hidden one: IEEE754 standard representation with hidden one
                        s - subnormals: IEEE754 subnormal numbers
            sign   - Static sign representation (exclusive option):
                        - IEEE754 dynamic sign is used if omitted
                        1 - all values are considered as negative numbers
                        0 - all values are considered as positive numbers

--fp-format=inline-math
    The "inline-math" flag may be added to fp-format option to force floating-point
    arithmetic operators always inline policy

--fp-format=inline-conversion
    The "inline-conversion" flag may be added to fp-format option to force floating-point
    conversion operators always inline policy

--fp-format-interface
    User-defined floating-point format is applied to top interface signature if required
    (default modifies top function body only)

--fp-format-propagate
    Propagate user-defined floating-point format to called function when possible
```

## Sine Wave Generator Application
The sine wave generator application is a very simple toy example to show the capabilities of the TrueFloat library embedded into the Bambu HLS tool.
```
Usage: sine_gen <samples> <frequency> <amplitude> <sampling rate> <out_file>
```

The application generates a set of points sampling a sine wave with given frequency and amplitude to show the effects of different numerical representations on the accelerator performance and on the output results.

## Generating the baseline
To generate a basic version of the hardware design it is necessary to specify just a few options:
 - Input file: *sine_table_gen.cpp*
 - Top function name: *--top-fname=sine_table_gen*
 - Lib Math linking: _-lm_
 - Testbench program: *--generate-tb=sine_gen.cpp*
 - Testbench program arguments: *--tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=baseline.dat*

In [ ]:
%mkdir -p /content/truefloat/baseline
%cd /content/truefloat/baseline
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=baseline.dat --simulate --print-dot |& tee log.txt

Now we may run the simulation to get the baseline result

In [ ]:
%cd /content/truefloat
from multi_plot import main as plot_waves
plot_waves(["baseline/baseline.dat"])

## Generate TrueFloat custom floating-point variants
After the baseline has been generated you may generate some variants with a custom precision floating-point format of your choice passing the `--fp-format` command line option.

As an example, we can generate the following variants:
- e5m5b-15nih
- e4m4b-7nih
- e5m3b-15nih

Since the above custom formats are far from comparable in precision with the original double precision representation, it may be useful to disable the embedded automatic verification against the input representation, since it will be way off the golden values. To do so use the `-DCUSTOM_VERIFICATION` command line option.

In [ ]:
%mkdir -p /content/truefloat/e5m5b_15nih
%cd /content/truefloat/e5m5b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m5b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m5b_15nih.dat --simulate 

In [ ]:
%mkdir -p /content/truefloat/e4m4b_7nih
%cd /content/truefloat/e4m4b_7nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e4m4b-7nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e4m4b_7nih.dat --simulate --print-dot |& tee log.txt

In [ ]:
%mkdir -p /content/truefloat/e5m3b_15nih
%cd /content/truefloat/e5m3b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m3b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m3b_15nih.dat --simulate --print-dot |& tee log.txt

## Look at the output
After the different variants have been generated, it is time to have a look at the generated output to verify the impact of the different custom floating-point formats on the application results.

In [ ]:
from pathlib import Path

base_dir = Path("/content/truefloat")
dat_files = []

for subdir in base_dir.iterdir():
    if subdir.is_dir():
        dat_files.extend(subdir.glob("*.dat"))

plot_waves(dat_files)